# C3 exp01: zero-shot prompting, k = 0, definitions on

Qwen2.5-0.5B-Instruct labels every val segment from a prompt (no training), scored by the probabilities of the answer letters A–G.
Note: `tasks/C3-prompting-baseline/experiments/exp01_zero-shot-k0/NOTE.md`.

Temporary trial (`scripts/trial_prompting.py`); its numbers are **not reportable** (PROTOCOL §4.1).

Before running, in the notebook settings (right-hand panel):
- **Accelerator:** GPU **T4 x2** (one GPU is used)
- **Internet:** on (to clone the repo and download the model)
- **Input:** your private `smi-v3-meetings` dataset attached

The branch `C3/prompting-baseline` must be pushed: the notebook clones it from GitHub.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
Do **not** install `requirements.txt` here: it would replace Kaggle's own torch build.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1

## 3. Data folder
Found automatically: the folder that contains `source.json`, `train/`, `val/`, `test/`.

In [ ]:
import glob, json, os

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
print("source.json found at:", hits)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input : is the dataset attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))
print(json.load(open(hits[0])))

## 4. Quick check: 64 segments
Downloads the model (~1 GB, once), checks that everything runs on the GPU, and shows the speed. Takes about a minute.

In [ ]:
!PYTHONPATH=src python scripts/trial_prompting.py --data-root {DATA_ROOT} --out-dir /kaggle/working/runs     --run-name c3_check --limit 64 --batch-size 32

## 5. exp01: the full val split (16,289 segments)
Zero-shot, k = 0, definitions on. Prints progress every 50 batches. If the session stops, run this cell again: it continues where it stopped.

In [ ]:
!PYTHONPATH=src python scripts/trial_prompting.py --data-root {DATA_ROOT} --out-dir /kaggle/working/runs     --run-name c3_exp01_zs_k0_defs --options definitions_v1.txt --past-k 0 --future-k 0 --shots-per-class 0 --batch-size 32

## 6. Results

In [ ]:
m = json.load(open("/kaggle/working/runs/c3_exp01_zs_k0_defs/metrics.json"))
print("data version :", m["data_version"], "| segments:", m["n_segments"])
print("macro-F1 (6 targets):", round(m["primary"]["macro_f1_6"], 4))
print("prompting    :", m["prompting"])
print()
print(f"{'class':6} {'P':>6} {'R':>6} {'F1':>6} {'support':>8}")
for label, c in m["per_class"].items():
    print(f"{label:6} {c['p']:6.3f} {c['r']:6.3f} {c['f1']:6.3f} {c['support']:8d}")
print()
print("confusion (rows = gold, columns = predicted):", m["confusion"]["labels"])
for label, row in zip(m["confusion"]["labels"], m["confusion"]["counts"]):
    print(f"{label:6}", row)

## 7. Download the outputs
Then in the right-hand panel: **Output** → `/kaggle/working` → `c3_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_runs.zip runs
!ls -la c3_runs.zip